In [ ]:
print("hello")

The patool library is helpful because it automatically detects which extraction tool (7-Zip, unrar, etc.) is available on your machine

In [ ]:
%pip install patool #  Extract .rar file

In [ ]:
from pathlib import Path

# Get the current working directory as a Path object
current_path = Path.cwd()
print(f"Current Path: {current_path}")


In [ ]:
import patoolib
from pathlib import Path

source_path = "data.rar" # assigning the source file name


current_path = Path.cwd() # current working directory
destination_folder = current_path / "data" # defining destination folder

print(destination_folder)

# patool will handle finding the right backend tool for you
patoolib.extract_archive(source_path, outdir=destination_folder) # extracts rar file to data folder


In [ ]:
%pip install pyyaml pandas # Extract .yaml file


In [ ]:
import os # to use os.walk 
import yaml # to use yaml.safe_Load fun 
import pandas as pd # to use data cleaning

def yaml_to_dataframe(root_directory):
    all_data = []

    # Using for loop to walk through the "data" directory
    for root, dirs, files in os.walk(root_directory):
        for file in files:
            if file.endswith(('.yaml', '.yml')):
                file_path = os.path.join(root, file)
                
                try:
                    with open(file_path, 'r') as f:
                        # Use safe_load to avoid executing arbitrary code
                        data = yaml.safe_load(f)
                        
                        if isinstance(data, dict):
                            # Optional: add metadata to track source file
                            data['_filename'] = file
                            all_data.append(data)
                        elif isinstance(data, list):
                            # Extend list if the YAML file itself contains a list of records
                            all_data.extend(data)
                            
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")

    # Create the DataFrame from the collected list of dictionaries
    return pd.DataFrame(all_data)

# Usage
target_folder = 'data'
stockpricedf = yaml_to_dataframe(target_folder)

# Display the first few rows
print(stockpricedf.head())


In [ ]:
import pandas as pd
import os

# Assuming your dataframe is named 'df'
# Create a folder to store the output files if it doesn't exist
output_dir = 'ticker_data'
os.makedirs(output_dir, exist_ok=True)

# Group by the 'Ticker' column and save each group
for ticker, data in stockpricedf.groupby('Ticker'):
    # Clean ticker name for filename (remove special characters if necessary)
    filename = f"{ticker}.csv"
    filepath = os.path.join(output_dir, filename)
    
    # Save to CSV (index=False avoids adding the row numbers to the file)
    data.to_csv(filepath, index=False)
    print(f"Saved: {filepath}")


In [ ]:
import pandas as pd

def get_top_green_stocks(df):
    # 1. Ensure date is in datetime format
    df['date'] = pd.to_datetime(df['date'])
    
    # 2. Sort by Ticker and Date to ensure chronological order
    df = df.sort_values(['Ticker', 'date'])
    
    # 3. Group by Ticker to get the first (opening) and last (closing) price
    summary = df.groupby('Ticker').agg(
        start_price=('close', 'first'),
        end_price=('close', 'last')
    )
    
    # 4. Calculate Percentage Return
    summary['return_pct'] = ((summary['end_price'] - summary['start_price']) / summary['start_price']) * 100
    
    # 5. Sort by return and take the top 10
    top_10 = summary.sort_values(by='return_pct', ascending=False).head(10)
    
    return top_10

# Usage:
# Assuming 'df' is your existing 14200-row DataFrame
top_10_performers = get_top_green_stocks(stockpricedf)
print(top_10_performers)


In [ ]:
%pip install pandas sqlalchemy pymysql


In [ ]:
%pip install plotly


In [ ]:
%pip install seaborn


In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Page configuration for a wide dashboard layout
st.set_page_config(layout="wide", page_title="Stock Performance Dashboard")


@st.cache_data
def load_and_merge_data():
    try:
        df = pd.read_csv('FullData.csv')
        sector_df = pd.read_csv('sectorData.csv')
        
        # 1. Clean column names
        df.columns = df.columns.str.strip()
        sector_df.columns = sector_df.columns.str.strip()

        # 2. SPLIT 'Symbol' column into 'Company_Name' and 'Ticker'
        # This handles the "ADANI ENTERPRISES: ADANIGREEN" format
        if 'Symbol' in sector_df.columns:
            split_data = sector_df['Symbol'].str.split(':', expand=True)
            sector_df['Company_Name'] = split_data[0].str.strip()
            sector_df['Ticker'] = split_data[1].str.strip()
            
            # Update 'Symbol' to only contain the Ticker for easier merging
            sector_df['Symbol'] = sector_df['Ticker']
        
        # 3. Handle Date column
        date_col = next((c for c in df.columns if c.lower() == 'date'), None)
        if date_col:
            df['Date'] = pd.to_datetime(df[date_col])
            df['Month_Year'] = df['Date'].dt.to_period('M').astype(str)
        
        # 4. Merge
        return df.merge(sector_df, on='Ticker', how='left')
        
    except Exception as e:
        st.error(f"Error processing data: {e}")
        st.stop()


# Initialize Data
df = load_and_merge_data()

# --- SIDEBAR FILTERS ---
st.sidebar.header("Dashboard Filters")

# Sector Filter
all_sectors = sorted(df['sector'].dropna().unique().tolist())
selected_sectors = st.sidebar.multiselect("Select Sectors", options=all_sectors, default=all_sectors)

# --- DATA PROCESSING ---
filtered_df = df[df['Sector'].isin(selected_sectors)]

# Calculate monthly returns (First price of month vs Last price of month)
monthly_stats = filtered_df.groupby(['Ticker', 'Month_Year', 'sector'])['close'].agg(['first', 'last']).reset_index()
monthly_stats['Returns'] = ((monthly_stats['last'] - monthly_stats['first']) / monthly_stats['first']) * 100

# Get exactly the last 12 months for the "12 Charts" requirement
all_months = sorted(monthly_stats['Month_Year'].unique(), reverse=True)[:12]

# --- MAIN DASHBOARD DISPLAY ---
st.title("📊 Top 5 Monthly Gainers & Losers (12-Month View)")

if not all_months:
    st.warning("No data found for the selected sectors.")
else:
    # Responsive grid: 3 columns x 4 rows to make 12 charts
    cols_per_row = 3
    for i in range(0, len(all_months), cols_per_row):
        batch = all_months[i:i + cols_per_row]
        st_cols = st.columns(len(batch))
        
        for idx, month in enumerate(batch):
            month_data = monthly_stats[monthly_stats['Month_Year'] == month]
            
            # Identify Top 5 Gainers and Top 5 Losers
            top_5 = month_data.nlargest(5, 'Returns')
            bottom_5 = month_data.nsmallest(5, 'Returns')
            plot_df = pd.concat([top_5, bottom_5]).sort_values('Returns', ascending=True)

            # Visualization
            fig, ax = plt.subplots(figsize=(8, 5))
            # Green for positive returns, Red for negative
            colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in plot_df['Returns']]
            
            sns.barplot(x='Returns', y='Symbol', data=plot_df, palette=colors, ax=ax)
            ax.set_title(f"Performance: {month}", fontsize=12, fontweight='bold')
            ax.axvline(0, color='black', linewidth=1)
            ax.set_xlabel("% Return")
            
            st_cols[idx].pyplot(fig)
            plt.close()

st.sidebar.info("This dashboard automatically visualizes the last 12 months of performance.")


In [ ]:
%pip install mysql-connector-python


In [ ]:
import mysql.connector
from mysql.connector import Error

def create_db_connection():
    """Establishes a connection to the local MySQL server."""
    try:
        connection = mysql.connector.connect(
            host="localhost",      # Local machine address
            user="root",   # Default is often 'root'
            password="your_password",
            database="tharani" # Optional: specify a starting database
        )
        
        if connection.is_connected():
            print("Successfully connected to the database")
            return connection

    except Error as e:
        print(f"Error while connecting to MySQL: {e}")
        return None

# Usage
conn = create_db_connection()

# Remember to close the connection when finished
if conn and conn.is_connected():
    conn.close()
    print("MySQL connection is closed")


In [ ]:
pip install sqlalchemy
